# LeetCode 1457 — Pseudo-Palindromic Paths in a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/pseudo-palindromic-paths-in-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def pseudoPalindromicPaths (self, root: TreeNode | None) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu điều kiện **"một hoán vị là palindrome"** ⇔ **tối đa một chữ số xuất hiện lẻ lần**.
2. Duyệt mọi **đường đi gốc → lá** bằng DFS, nhận diện **lá** đúng cách (không có cả hai con).
3. Duy trì tần suất bằng mảng đếm có **quay lui (backtracking)**, rồi nén thành **bitmask chẵn lẻ** với `mask ^= 1 << val`.
4. Dùng mẹo bit `mask & (mask − 1) == 0` để kiểm tra "mask có ≤ 1 bit bật".
5. Tránh lỗi: đếm ở nút `None` hay nút một con, đòi mọi tần suất chẵn, quên quay lui, nhầm `mask ^= val`.

## 2. Đề bài & ví dụ

Cho cây nhị phân, giá trị mỗi nút là chữ số từ `1` đến `9`. Một đường đi được gọi là **giả palindrome (pseudo-palindromic)** nếu **có ít nhất một hoán vị** của các giá trị trên đường đi là palindrome.

Đếm số đường đi **từ gốc tới lá** là giả palindrome.

**Ví dụ 1:** `root = [2,3,1,3,1,null,1]` → `2`. Ba đường: `[2,3,3]` (hoán vị `3,2,3` ✔), `[2,1,1]` (`1,2,1` ✔), `[2,3,1]` ✘.

```
        2
      /   \
     3     1
    / \     \
   3   1     1
```

**Ví dụ 2:** `root = [2,1,1,1,3,null,null,null,null,null,1]` → `1`. Ba đường `[2,1,1]` ✔, `[2,1,3,1]` ✘, `[2,1]` ✘.

**Ví dụ 3:** `root = [9]` → `1`.

**Ràng buộc:** số nút trong `[1, 10^5]`; `1 <= Node.val <= 9`.

In [1]:
import random
import sys
import time
from collections import Counter, deque, defaultdict
from itertools import permutations
from typing import List, Optional

examples = [
    ([2, 3, 1, 3, 1, None, 1], 2),
    ([2, 1, 1, 1, 3, None, None, None, None, None, 1], 1),
    ([9], 1),
]
for arr, want in examples:
    print(f"root={arr} -> kỳ vọng {want}")

root=[2, 3, 1, 3, 1, None, 1] -> kỳ vọng 2
root=[2, 1, 1, 1, 3, None, None, None, None, None, 1] -> kỳ vọng 1
root=[9] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 `TreeNode` và dựng cây từ level-order

Tự dựng lớp nút và hàm `build_tree` (hàng đợi gán con trái/phải theo mảng, `None` là chỗ trống) để chạy thử trên máy.

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def root_to_leaf_paths(root):
    out, stack = [], [(root, [root.val])]
    while stack:
        node, path = stack.pop()
        if not node.left and not node.right:
            out.append(path)
        for c in (node.right, node.left):
            if c:
                stack.append((c, path + [c.val]))
    return out

print(root_to_leaf_paths(build_tree([2, 3, 1, 3, 1, None, 1])))
print(root_to_leaf_paths(build_tree([2, 1, 1, 1, 3, None, None, None, None, None, 1])))

[[2, 3, 3], [2, 3, 1], [2, 1, 1]]
[[2, 1, 1], [2, 1, 3, 1], [2, 1]]


### 3.2 Khi nào một dãy có hoán vị là palindrome?

Palindrome đọc xuôi ngược như nhau ⇒ các ký tự **ghép cặp đối xứng**; chỉ ký tự ở **giữa** (nếu độ dài lẻ) được đứng một mình. Vậy: **có hoán vị palindrome ⇔ số giá trị xuất hiện lẻ lần ≤ 1**. Kiểm chứng bằng cách thử mọi hoán vị (chỉ với dãy ngắn):

In [3]:
def has_pal_perm_brute(seq):
    return any(list(p) == list(p)[::-1] for p in set(permutations(seq)))

def has_pal_perm(seq):
    return sum(v % 2 for v in Counter(seq).values()) <= 1

for seq in ([2, 3, 3], [2, 1, 1], [2, 3, 1], [1, 2, 1, 2], [1, 2, 3, 3, 2], [7]):
    print(seq, Counter(seq), "| thử hoán vị:", has_pal_perm_brute(seq), "| đếm lẻ:", has_pal_perm(seq))

[2, 3, 3] Counter({3: 2, 2: 1}) | thử hoán vị: True | đếm lẻ: True
[2, 1, 1] Counter({1: 2, 2: 1}) | thử hoán vị: True | đếm lẻ: True
[2, 3, 1] Counter({2: 1, 3: 1, 1: 1}) | thử hoán vị: False | đếm lẻ: False
[1, 2, 1, 2] Counter({1: 2, 2: 2}) | thử hoán vị: True | đếm lẻ: True
[1, 2, 3, 3, 2] Counter({2: 2, 3: 2, 1: 1}) | thử hoán vị: True | đếm lẻ: True
[7] Counter({7: 1}) | thử hoán vị: True | đếm lẻ: True


### 3.3 XOR và bitmask chẵn lẻ

Chỉ cần **tính chẵn lẻ** của từng chữ số, không cần số lần chính xác. Dùng 1 bit cho mỗi chữ số `d`: `mask ^= 1 << d` **lật** bit `d` mỗi khi gặp `d`. Bit `d` bật ⇔ `d` xuất hiện lẻ lần.

In [4]:
mask = 0
for d in [2, 3, 3]:
    mask ^= 1 << d
    print(f"gặp {d}: mask = {mask:010b}")
print("các chữ số lẻ lần:", [d for d in range(10) if mask >> d & 1])

gặp 2: mask = 0000000100
gặp 3: mask = 0000001100
gặp 3: mask = 0000000100
các chữ số lẻ lần: [2]


### 3.4 Mẹo `x & (x − 1)` — xoá bit thấp nhất

`x − 1` lật bit 1 thấp nhất của `x` và mọi bit 0 phía dưới nó ⇒ `x & (x − 1)` là `x` bỏ đi bit 1 thấp nhất. Do đó `x & (x − 1) == 0` ⇔ `x` có **không quá 1 bit** bật (kể cả `x = 0`). Có thể kiểm tra bằng `bin(x).count("1") <= 1` hoặc `x.bit_count()` (Python 3.10+).

In [5]:
for x in [0, 0b1000, 0b1100, 0b10000001, 1 << 9]:
    print(f"x={x:010b}  x&(x-1)={x & (x - 1):010b}  ≤1 bit? {x & (x - 1) == 0}  (đếm bit: {bin(x).count('1')})")

x=0000000000  x&(x-1)=0000000000  ≤1 bit? True  (đếm bit: 0)
x=0000001000  x&(x-1)=0000000000  ≤1 bit? True  (đếm bit: 1)
x=0000001100  x&(x-1)=0000001000  ≤1 bit? False  (đếm bit: 2)
x=0010000001  x&(x-1)=0010000000  ≤1 bit? False  (đếm bit: 2)
x=1000000000  x&(x-1)=0000000000  ≤1 bit? True  (đếm bit: 1)


### 3.5 Quay lui (backtracking) với trạng thái dùng chung

Khi DFS dùng **một** mảng đếm chung: tăng khi đi xuống nút, **giảm lại** khi rời nút. Như vậy lúc ở nút nào, mảng phản ánh đúng đường đi từ gốc tới nút đó.

In [6]:
cnt = [0] * 10
def demo(node, depth=0):
    if node is None:
        return
    cnt[node.val] += 1
    print("  " * depth + f"vào {node.val}: cnt[1..4] = {cnt[1:5]}")
    demo(node.left, depth + 1)
    demo(node.right, depth + 1)
    cnt[node.val] -= 1                      # quay lui
    print("  " * depth + f"rời {node.val}: cnt[1..4] = {cnt[1:5]}")

demo(build_tree([2, 3, 1]))

vào 2: cnt[1..4] = [0, 1, 0, 0]
  vào 3: cnt[1..4] = [0, 1, 1, 0]
  rời 3: cnt[1..4] = [0, 1, 0, 0]
  vào 1: cnt[1..4] = [1, 1, 0, 0]
  rời 1: cnt[1..4] = [0, 1, 0, 0]
rời 2: cnt[1..4] = [0, 0, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1. Với mỗi đường gốc → lá, đếm tần suất và số chữ số lẻ lần:

| đường | tần suất | chữ số lẻ lần | ≤ 1? |
|---|---|---|---|
| 2 → 3 → 3 | {2:1, 3:2} | {2} | ✔ |
| 2 → 3 → 1 | {2:1, 3:1, 1:1} | {1,2,3} | ✘ |
| 2 → 1 → 1 | {2:1, 1:2} | {2} | ✔ |

Kết quả 2. Hai quan sát dẫn tới lời giải hiệu quả:
1. Đường đi tới các lá **chung tiền tố** (đoạn gần gốc) ⇒ không nên xây lại từ đầu cho từng lá; DFS mang trạng thái xuống.
2. Chỉ cần **chẵn/lẻ** ⇒ 9 bit là đủ để mô tả trạng thái.

In [7]:
for path in root_to_leaf_paths(build_tree([2, 3, 1, 3, 1, None, 1])):
    odd = [d for d, c in Counter(path).items() if c % 2]
    print(f"đường {path}: lẻ lần {odd} -> {'✔' if len(odd) <= 1 else '✘'}")

đường [2, 3, 3]: lẻ lần [2] -> ✔
đường [2, 3, 1]: lẻ lần [2, 3, 1] -> ✘
đường [2, 1, 1]: lẻ lần [2] -> ✔


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** liệt kê **toàn bộ** đường gốc → lá (mỗi đường là một list riêng), với mỗi đường dùng `Counter` kiểm tra số giá trị lẻ lần ≤ 1.

In [8]:
def pseudo_brute(root: Optional[TreeNode]) -> int:
    return sum(has_pal_perm(path) for path in root_to_leaf_paths(root))

for arr, want in examples:
    print(pseudo_brute(build_tree(arr)), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
1 kỳ vọng 1


**Độ phức tạp:** mỗi phần tử stack mang một bản sao đường đi ⇒ Time và Space `O(n·h)` (cây lệch: `O(n²)`). Chưa kể `Counter` cho mỗi lá tốn `O(h)`.

**Nút thắt:** sao chép và đếm lại cả đường đi cho từng nút/lá. Trạng thái thực sự cần chỉ là **9 bit chẵn lẻ**.

In [9]:
def chain(n, rng):
    # cây lệch phải n nút, giá trị ngẫu nhiên 1..9
    arr = [rng.randint(1, 9)]
    for _ in range(n - 1):
        arr += [None, rng.randint(1, 9)]
    return build_tree(arr)

rng = random.Random(0)
t = time.perf_counter()
print(pseudo_brute(chain(4000, rng)), f"(brute, cây lệch 4000 nút: {time.perf_counter()-t:.2f}s)")

0 (brute, cây lệch 4000 nút: 0.03s)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — điều kiện palindrome:** đường đi tốt ⇔ số chữ số có tần suất lẻ ≤ 1.

**Insight 2 — trạng thái đi xuống:** DFS từ gốc, mang theo trạng thái của đường đi hiện tại; tại **lá** (không có con trái **và** không có con phải) kiểm tra điều kiện.

**Insight 3 — bitmask:** trạng thái = `mask` 10 bit, `mask ^= 1 << val` khi vào nút. Vì `mask` là **số nguyên bất biến** truyền theo tham số, mỗi nhánh có bản riêng — **không cần quay lui**.

**Quy tắc:** **nếu** nút là lá **và** `mask & (mask − 1) == 0` **thì** đếm 1.

**Invariant:** khi xử lý nút `u` (sau khi lật bit của `u`), bit `d` của `mask` bật ⇔ chữ số `d` xuất hiện lẻ lần trên đường gốc → `u`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS + mảng đếm 9 chữ số + đếm số lẻ, có quay lui

**Giải thích từng đoạn:**
1. `cnt[d]` = số lần `d` trên đường hiện tại; `odd` = số chữ số đang lẻ.
2. Vào nút: `cnt[v] += 1`, cập nhật `odd += 1` nếu `cnt[v]` thành lẻ, ngược lại `odd −= 1`.
3. Lá: cộng `odd <= 1`.
4. Rời nút: hoàn tác chính xác hai thay đổi trên.

In [10]:
def pseudo_count(root: Optional[TreeNode]) -> int:
    cnt = [0] * 10
    state = {"odd": 0, "ans": 0}

    def dfs(node):
        v = node.val
        cnt[v] += 1
        state["odd"] += 1 if cnt[v] % 2 else -1
        if not node.left and not node.right:
            state["ans"] += state["odd"] <= 1
        else:
            if node.left:
                dfs(node.left)
            if node.right:
                dfs(node.right)
        cnt[v] -= 1                                   # quay lui
        state["odd"] += 1 if cnt[v] % 2 else -1

    dfs(root)
    return state["ans"]

for arr, want in examples:
    print(pseudo_count(build_tree(arr)), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
1 kỳ vọng 1


Khi giảm `cnt[v]`: nếu `cnt[v]` mới lẻ thì trước đó chẵn ⇒ số lẻ tăng 1; nếu mới chẵn thì trước đó lẻ ⇒ số lẻ giảm 1 — cùng công thức với lúc tăng.

**Code nộp LeetCode (cách 2):**

In [11]:
class SolutionCount:
    def pseudoPalindromicPaths(self, root: Optional[TreeNode]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 2 * 10**5 + 10))
        cnt = [0] * 10
        self.odd = self.ans = 0

        def dfs(node):
            v = node.val
            cnt[v] += 1
            self.odd += 1 if cnt[v] & 1 else -1
            if not node.left and not node.right:
                self.ans += self.odd <= 1
            for c in (node.left, node.right):
                if c:
                    dfs(c)
            cnt[v] -= 1
            self.odd += 1 if cnt[v] & 1 else -1

        dfs(root)
        return self.ans

print(SolutionCount().pseudoPalindromicPaths(build_tree([2, 3, 1, 3, 1, None, 1])))

2


**Độ phức tạp:** Time `O(n)` (mỗi nút `O(1)`), Space `O(h)` stack đệ quy + mảng 10 phần tử.

### 7.2 Cách 3 — DFS đệ quy với bitmask XOR

**Giải thích từng đoạn:**
1. `dfs(node, mask)`: lật bit `node.val`.
2. Lá ⇒ trả về `1 if mask & (mask − 1) == 0 else 0`.
3. Không phải lá ⇒ cộng kết quả của các con (truyền `mask` mới — không cần hoàn tác).

In [12]:
def pseudo_mask(root: Optional[TreeNode]) -> int:
    def dfs(node, mask):
        if node is None:
            return 0
        mask ^= 1 << node.val
        if not node.left and not node.right:
            return 1 if mask & (mask - 1) == 0 else 0
        return dfs(node.left, mask) + dfs(node.right, mask)
    return dfs(root, 0)

for arr, want in examples:
    print(pseudo_mask(build_tree(arr)), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
1 kỳ vọng 1


**Code nộp LeetCode (cách 3):**

In [13]:
class SolutionMask:
    def pseudoPalindromicPaths(self, root: Optional[TreeNode]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 2 * 10**5 + 10))

        def dfs(node, mask):
            if node is None:
                return 0
            mask ^= 1 << node.val
            if node.left is None and node.right is None:
                return int(mask & (mask - 1) == 0)
            return dfs(node.left, mask) + dfs(node.right, mask)

        return dfs(root, 0)

print(SolutionMask().pseudoPalindromicPaths(build_tree([2, 1, 1, 1, 3, None, None, None, None, None, 1])))

1


**Độ phức tạp:** Time `O(n)`, Space `O(h)`.

### 7.3 Cách 4 — DFS lặp bằng stack `(nút, mask)` (không đệ quy)

**Giải thích:** mỗi phần tử stack mang `mask` **của đường tới cha**; lấy ra thì lật bit của nút, kiểm tra lá, đẩy các con. An toàn với cây lệch `10^5` tầng. Dùng `deque.popleft` thay `pop` sẽ thành BFS — kết quả như nhau.

In [14]:
def pseudo_stack(root: Optional[TreeNode]) -> int:
    ans = 0
    stack = [(root, 0)]
    while stack:
        node, mask = stack.pop()
        mask ^= 1 << node.val
        if not node.left and not node.right:
            ans += mask & (mask - 1) == 0
        else:
            if node.left:
                stack.append((node.left, mask))
            if node.right:
                stack.append((node.right, mask))
    return ans

def pseudo_bfs(root: Optional[TreeNode]) -> int:
    ans, q = 0, deque([(root, 0)])
    while q:
        node, mask = q.popleft()
        mask ^= 1 << node.val
        if not node.left and not node.right:
            ans += mask & (mask - 1) == 0
        q.extend((c, mask) for c in (node.left, node.right) if c)
    return ans

for arr, want in examples:
    print(pseudo_stack(build_tree(arr)), pseudo_bfs(build_tree(arr)), "kỳ vọng", want)

2 2 kỳ vọng 2
1 1 kỳ vọng 1
1 1 kỳ vọng 1


**Code nộp LeetCode (cách 4)** — xem mục 9 (lời giải chính). **Độ phức tạp:** Time `O(n)`, Space `O(h)`–`O(n)`.

So sánh tốc độ trên cây `10^5` nút (bỏ brute force; cây lệch bỏ các bản đệ quy):

In [15]:
def random_tree(n, rng):
    root = TreeNode(rng.randint(1, 9))
    slots = [(root, 0), (root, 1)]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        node = TreeNode(rng.randint(1, 9))
        if side == 0:
            parent.left = node
        else:
            parent.right = node
        slots += [(node, 0), (node, 1)]
    return root

rng = random.Random(1457)
big_random = random_tree(10**5, rng)
big_chain = chain(10**5, rng)
for tname, tree in [("ngẫu nhiên", big_random), ("lệch phải", big_chain)]:
    for name, f in [("count", pseudo_count), ("mask", pseudo_mask), ("stack", pseudo_stack), ("bfs", pseudo_bfs)]:
        if tname == "lệch phải" and name in ("count", "mask"):
            continue      # đệ quy 10^5 tầng: không chạy trong notebook
        t = time.perf_counter()
        r = f(tree)
        print(f"{tname:10} {name:6} -> {r:6d} ({time.perf_counter()-t:.3f}s)")

ngẫu nhiên count  ->    696 (0.039s)
ngẫu nhiên mask   ->    696 (0.024s)
ngẫu nhiên stack  ->    696 (0.028s)
ngẫu nhiên bfs    ->    696 (0.067s)
lệch phải  stack  ->      0 (0.014s)
lệch phải  bfs    ->      0 (0.039s)


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1. Bit được in theo thứ tự chữ số `9…1` (bỏ bit 0 không dùng).

| lấy ra | mask cũ (của cha) | lật bit | mask mới | lá? | ≤1 bit? | ans |
|---|---|---|---|---|---|---|
| 2 | 0 | 2 | {2} | không | | 0 |
| 1 (phải) | {2} | 1 | {1,2} | không | | 0 |
| 1 (lá) | {1,2} | 1 | {2} | có | ✔ | 1 |
| 3 | {2} | 3 | {2,3} | không | | 1 |
| 1 (lá) | {2,3} | 1 | {1,2,3} | có | ✘ | 1 |
| 3 (lá) | {2,3} | 3 | {2} | có | ✔ | 2 |

In [16]:
def fmt(mask):
    return format(mask >> 1, "09b")        # chữ số 9..1

def trace_stack(root):
    ans, stack = 0, [(root, 0)]
    while stack:
        node, mask = stack.pop()
        new = mask ^ (1 << node.val)
        leaf = not node.left and not node.right
        ok = new & (new - 1) == 0
        if leaf and ok:
            ans += 1
        print(f"nút {node.val}: mask {fmt(mask)} -> {fmt(new)} | lẻ lần {[d for d in range(1, 10) if new >> d & 1]}"
              f" | {'lá, ' + ('✔' if ok else '✘') if leaf else 'không phải lá'} | ans={ans}")
        for c in (node.left, node.right):
            if c:
                stack.append((c, new))
    return ans

trace_stack(build_tree([2, 3, 1, 3, 1, None, 1]))

nút 2: mask 000000000 -> 000000010 | lẻ lần [2] | không phải lá | ans=0
nút 1: mask 000000010 -> 000000011 | lẻ lần [1, 2] | không phải lá | ans=0
nút 1: mask 000000011 -> 000000010 | lẻ lần [2] | lá, ✔ | ans=1
nút 3: mask 000000010 -> 000000110 | lẻ lần [2, 3] | không phải lá | ans=1
nút 1: mask 000000110 -> 000000111 | lẻ lần [1, 2, 3] | lá, ✘ | ans=1
nút 3: mask 000000110 -> 000000010 | lẻ lần [2] | lá, ✔ | ans=2


2

Trace cách 2 (mảng đếm + quay lui) trên ví dụ 2, in `odd` lúc vào/rời từng nút:

In [17]:
def trace_count(root):
    cnt, info = [0] * 10, {"odd": 0, "ans": 0}
    def dfs(node, d):
        v = node.val
        cnt[v] += 1
        info["odd"] += 1 if cnt[v] % 2 else -1
        leaf = not node.left and not node.right
        if leaf:
            info["ans"] += info["odd"] <= 1
        print("  " * d + f"vào {v}: odd={info['odd']}" + (f" | LÁ -> ans={info['ans']}" if leaf else ""))
        for c in (node.left, node.right):
            if c:
                dfs(c, d + 1)
        cnt[v] -= 1
        info["odd"] += 1 if cnt[v] % 2 else -1
        print("  " * d + f"rời {v}: odd={info['odd']}")
    dfs(root, 0)
    return info["ans"]

trace_count(build_tree([2, 1, 1, 1, 3, None, None, None, None, None, 1]))

vào 2: odd=1
  vào 1: odd=2
    vào 1: odd=1 | LÁ -> ans=1
    rời 1: odd=2
    vào 3: odd=3
      vào 1: odd=2 | LÁ -> ans=1
      rời 1: odd=3
    rời 3: odd=2
  rời 1: odd=1
  vào 1: odd=2 | LÁ -> ans=1
  rời 1: odd=1
rời 2: odd=0


1

## 9. Code nộp LeetCode

Lời giải chính (cách 4 — stack + bitmask, không đệ quy), đúng chữ ký `pseudoPalindromicPaths(self, root)`. Trên LeetCode `TreeNode` có sẵn; ở đây dùng lớp tự dựng phía trên.

In [18]:
class Solution:
    def pseudoPalindromicPaths(self, root: Optional[TreeNode]) -> int:
        ans = 0
        stack = [(root, 0)]
        while stack:
            node, mask = stack.pop()
            mask ^= 1 << node.val
            if node.left is None and node.right is None:
                if mask & (mask - 1) == 0:
                    ans += 1
            else:
                if node.left:
                    stack.append((node.left, mask))
                if node.right:
                    stack.append((node.right, mask))
        return ans

for arr, want in examples:
    assert Solution().pseudoPalindromicPaths(build_tree(arr)) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

1. **Điều kiện palindrome:** nếu dãy có hoán vị palindrome `p`, thì `p[i] = p[L−1−i]` ghép các vị trí thành cặp cùng giá trị, trừ vị trí giữa khi `L` lẻ ⇒ mọi giá trị có tần suất chẵn, trừ tối đa một. Ngược lại, nếu tối đa một giá trị lẻ lần: xếp mỗi giá trị `c/2` lần vào nửa trái, đối xứng sang nửa phải, giá trị lẻ (nếu có) đặt giữa ⇒ palindrome.
2. **Bitmask:** XOR là phép lật; lật bit `d` đúng `c_d` lần cho bit bằng `c_d mod 2`. Vậy `mask` mã hoá đúng tập chữ số lẻ lần, và "≤ 1 giá trị lẻ" ⇔ `mask` có ≤ 1 bit ⇔ `mask & (mask − 1) == 0`.
3. **Duyệt:** mỗi lá được thăm đúng một lần, kèm `mask` của đường gốc → lá đó (quy nạp: con nhận `mask` của cha đã lật bit cha). Đường gốc → lá ↔ lá là song ánh ⇒ đếm đúng.
4. **Cách 2:** quay lui khôi phục `cnt` và `odd` về đúng trạng thái trước khi vào nút, nên khi sang nhánh anh em trạng thái phản ánh đúng đường đi.

## 11. Phân tích Time / Space

Gọi `n` là số nút, `h` là chiều cao.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute (liệt kê đường) | `O(n·h)` | `O(n·h)` | sao chép đường đi |
| 2. Mảng đếm + quay lui | `O(n)` | `O(h)` | phải hoàn tác cẩn thận |
| 3. Bitmask đệ quy | `O(n)` | `O(h)` | gọn, không cần hoàn tác |
| 4. Bitmask stack/BFS | `O(n)` | `O(h)`–`O(n)` | an toàn với cây sâu |

Trạng thái mỗi nút chỉ là một số nguyên 10 bit ⇒ chi phí hằng số.

## 12. Bộ test

Ví dụ đề, edge case (1 nút, toàn cùng chữ số, cây lệch, nút chỉ có một con — không phải lá), đối chiếu ngẫu nhiên với brute force và với kiểm tra **bằng hoán vị thật** trên cây nhỏ (seed cố định).

In [19]:
ALL = {
    "brute": pseudo_brute,
    "count": pseudo_count,
    "mask": pseudo_mask,
    "stack": pseudo_stack,
    "bfs": pseudo_bfs,
    "SolutionCount": lambda r: SolutionCount().pseudoPalindromicPaths(r),
    "SolutionMask": lambda r: SolutionMask().pseudoPalindromicPaths(r),
    "Solution": lambda r: Solution().pseudoPalindromicPaths(r),
}
edge = [
    ([1], 1),
    ([5, 5, 5, 5, None, None, 5], 2),
    ([1, 2], 0),                 # đường [1,2]: hai chữ số lẻ lần
    ([1, 1], 1),
    ([1, 2, None, 1], 1),        # [1,2,1]; nút 2 có 1 con nên không phải lá
    ([1, None, 2, None, 3], 0),
    ([2, 2, 2, 2, 2, 2, 2], 4),
]
for arr, want in examples + edge:
    for name, f in ALL.items():
        got = f(build_tree(arr))
        assert got == want, (name, arr, got, want)
print("Ví dụ đề + edge case: OK cho", len(ALL), "cách")

Ví dụ đề + edge case: OK cho 8 cách


In [20]:
def pseudo_perm_brute(root):
    # kiểm tra bằng hoán vị thật (chỉ dùng với đường ngắn)
    return sum(has_pal_perm_brute(p) for p in root_to_leaf_paths(root))

def small_random_tree(n, rng, digits):
    root = TreeNode(rng.choice(digits))
    slots = [(root, 0), (root, 1)]
    for _ in range(n - 1):
        parent, side = slots.pop(rng.randrange(len(slots)))
        node = TreeNode(rng.choice(digits))
        setattr(parent, "left" if side == 0 else "right", node)
        slots += [(node, 0), (node, 1)]
    return root

rng = random.Random(2024)
for _ in range(400):
    tree = small_random_tree(rng.randint(1, 9), rng, rng.choice([[1, 2], [1, 2, 3], list(range(1, 10))]))
    want = pseudo_perm_brute(tree)
    for name, f in ALL.items():
        assert f(tree) == want, name
print("400 cây nhỏ: mọi cách khớp kiểm tra bằng hoán vị thật")
for _ in range(300):
    tree = small_random_tree(rng.randint(1, 60), rng, rng.choice([[1, 2], [3, 5, 7], list(range(1, 10))]))
    want = pseudo_brute(tree)
    for name, f in ALL.items():
        assert f(tree) == want, name
print("300 cây vừa: mọi cách khớp brute force")
assert pseudo_stack(big_random) == pseudo_bfs(big_random) == Solution().pseudoPalindromicPaths(big_random)
print("Cây 10^5 nút: stack, bfs, Solution khớp nhau")

400 cây nhỏ: mọi cách khớp kiểm tra bằng hoán vị thật
300 cây vừa: mọi cách khớp brute force


Cây 10^5 nút: stack, bfs, Solution khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra ở nút `None` thay vì ở lá

Viết `if node is None: return check(mask)` ⇒ mỗi lá bị đếm **2 lần** (hai con `None`), và nút chỉ có một con cũng bị đếm (qua con `None` của nó) dù nó không phải lá.

In [21]:
def wrong_count_at_none(root):
    def dfs(node, mask):
        if node is None:
            return int(mask & (mask - 1) == 0)       # SAI
        mask ^= 1 << node.val
        return dfs(node.left, mask) + dfs(node.right, mask)
    return dfs(root, 0)

for arr in ([9], [2, 1, 1, 1, 3, None, None, None, None, None, 1], [1, 2, None, 1]):
    print(f"{arr}: Sai {wrong_count_at_none(build_tree(arr))} | Đúng {Solution().pseudoPalindromicPaths(build_tree(arr))}")

[9]: Sai 2 | Đúng 1
[2, 1, 1, 1, 3, None, None, None, None, None, 1]: Sai 2 | Đúng 1
[1, 2, None, 1]: Sai 3 | Đúng 1


**Sửa:** chỉ kiểm tra khi `node.left is None and node.right is None`.

### Lỗi 2 — Đòi **mọi** tần suất đều chẵn

Đường độ dài lẻ (như `[9]` hay `[2,3,3]`) luôn có đúng một chữ số lẻ lần và vẫn là palindrome được.

In [22]:
def wrong_all_even(root):
    ans, stack = 0, [(root, 0)]
    while stack:
        node, mask = stack.pop()
        mask ^= 1 << node.val
        if not node.left and not node.right:
            ans += mask == 0                         # SAI
        for c in (node.left, node.right):
            if c:
                stack.append((c, mask))
    return ans

for arr, _ in examples:
    print(f"{arr}: Sai {wrong_all_even(build_tree(arr))} | Đúng {Solution().pseudoPalindromicPaths(build_tree(arr))}")

[2, 3, 1, 3, 1, None, 1]: Sai 0 | Đúng 2
[2, 1, 1, 1, 3, None, None, None, None, None, 1]: Sai 0 | Đúng 1
[9]: Sai 0 | Đúng 1


**Sửa:** điều kiện `mask & (mask − 1) == 0` (0 hoặc 1 bit).

### Lỗi 3 — Mảng đếm dùng chung nhưng quên quay lui

Không giảm `cnt[v]` khi rời nút ⇒ nhánh phải "nhìn thấy" các chữ số của nhánh trái.

In [23]:
def wrong_no_backtrack(root):
    cnt = [0] * 10
    ans = [0]
    def dfs(node):
        cnt[node.val] += 1
        if not node.left and not node.right:
            ans[0] += sum(c % 2 for c in cnt) <= 1
        for c in (node.left, node.right):
            if c:
                dfs(c)
        # SAI: thiếu cnt[node.val] -= 1
    dfs(root)
    return ans[0]

for arr in ([2, 3, 1, 3, 1, None, 1], [2, 2, 3]):
    print(f"{arr}: Sai {wrong_no_backtrack(build_tree(arr))} | Đúng {Solution().pseudoPalindromicPaths(build_tree(arr))}")

[2, 3, 1, 3, 1, None, 1]: Sai 1 | Đúng 2
[2, 2, 3]: Sai 2 | Đúng 1


**Sửa:** `cnt[node.val] -= 1` trước khi `return` (hoặc dùng bitmask truyền theo tham số).

### Lỗi 4 — `mask ^= val` thay vì `mask ^= 1 << val`

XOR trực tiếp giá trị trộn lẫn các bit của nhiều chữ số (ví dụ `2 ^ 3 ^ 1 = 0`), không còn là "bit chẵn lẻ của từng chữ số".

In [24]:
def wrong_xor_value(root):
    ans, stack = 0, [(root, 0)]
    while stack:
        node, mask = stack.pop()
        mask ^= node.val                             # SAI
        if not node.left and not node.right:
            ans += mask & (mask - 1) == 0
        for c in (node.left, node.right):
            if c:
                stack.append((c, mask))
    return ans

arr = [2, 3, 1, 3, 1, None, 1]
print("Sai :", wrong_xor_value(build_tree(arr)), "| Đúng:", Solution().pseudoPalindromicPaths(build_tree(arr)))
print("Đường [2,3,1]: 2^3^1 =", 2 ^ 3 ^ 1, "-> bị coi là palindrome dù cả ba chữ số đều lẻ lần.")

Sai : 3 | Đúng: 2
Đường [2,3,1]: 2^3^1 = 0 -> bị coi là palindrome dù cả ba chữ số đều lẻ lần.


**Sửa:** `mask ^= 1 << node.val`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Hoán vị thành palindrome"** ⇒ đếm **số giá trị lẻ lần ≤ 1** (LeetCode 266 Palindrome Permutation).
- Chỉ cần **chẵn/lẻ** của ít loại giá trị ⇒ **bitmask + XOR**; kiểm tra "≤ 1 bit" bằng `x & (x − 1) == 0`.
- **Đường gốc → lá** ⇒ DFS mang trạng thái xuống; kiểm tra tại **lá thật** (`not left and not right`).
- Trạng thái bất biến (int) truyền theo tham số ⇒ khỏi quay lui; trạng thái dùng chung (list) ⇒ **phải** quay lui.
- Python + cây sâu `10^5` ⇒ stack lặp hoặc nâng recursion limit.

## 15. Biến thể của bài

1. **Liệt kê** các đường giả palindrome (bài tập 4).
2. **Đường đi xuống bất kỳ** (bắt đầu ở một nút, đi xuống tới một nút con cháu) — prefix mask + dict, giống "Path Sum III" (bài tập 5).
3. **Count Paths That Can Form a Palindrome in a Tree** (LeetCode 2791): đường giữa hai nút bất kỳ, ký tự trên cạnh — cùng ý tưởng mask + đếm.
4. **Longest Awesome Substring** (LeetCode 1542): chuỗi con dài nhất có hoán vị palindrome — prefix mask + vị trí đầu tiên.
5. **Palindrome Permutation** (LeetCode 266): phiên bản trên một chuỗi.

In [25]:
def can_permute_palindrome(s):
    # LeetCode 266 — cùng điều kiện, bitmask trên 26 chữ cái
    mask = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - 97)
    return mask & (mask - 1) == 0

print(can_permute_palindrome("carerac"), can_permute_palindrome("code"), can_permute_palindrome("aab"))

True False True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Liệt kê đường, `Counter` | `O(n·h)` | `O(n·h)` | Kiểm chứng |
| 2. Mảng đếm + quay lui | `cnt[10]`, `odd` | `O(n)` | `O(h)` | Khi chưa quen bit |
| 3. Bitmask đệ quy | `mask ^= 1 << v` | `O(n)` | `O(h)` | Gọn, phỏng vấn |
| 4. Bitmask stack/BFS | `(nút, mask)` | `O(n)` | `O(h)`–`O(n)` | Nộp bài Python an toàn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `pseudoPalindromicPaths([1,9,1,None,1,None,1,None,None,7,None,None,4])` = ?

**Bài 2 (tính tay):** với ví dụ 1, viết `mask` (dạng tập chữ số lẻ lần) tại mỗi lá.

**Bài 3 (điền TODO):** hoàn thiện `pseudo_todo` (DFS lặp + bitmask).

**Bài 4 (tự viết lại):** viết `list_pseudo_paths(root)` trả về danh sách các đường giả palindrome (mỗi đường là list giá trị gốc → lá), sắp tăng dần để so sánh.

**Bài 5 (mini challenge):** viết `count_downward(root)` đếm số đường **đi xuống** (bắt đầu ở nút `u` bất kỳ, kết thúc ở nút `v` là `u` hoặc con cháu của `u`, gồm cả hai đầu) là giả palindrome, `O(n·10)`.

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def pseudo_todo(root):
    ans, stack = 0, [(root, 0)]
    while stack:
        node, mask = stack.pop()
        # TODO: lật bit node.val trong mask
        # TODO: nếu node là lá và mask có <= 1 bit thì ans += 1
        # TODO: đẩy các con (khác None) kèm mask mới
        pass
    return ans

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Dựng cây bằng `build_tree` trong đầu: gốc 1, con 9 và 1. (2) Liệt kê đường gốc → lá. (3) Đếm chữ số lẻ lần trên từng đường.

**Bài 2:** (1) Lá là 3 (trái-trái), 1 (trái-phải), 1 (phải-phải). (2) Lật bit dọc đường. (3) `{2}`, `{1,2,3}`, `{2}`.

**Bài 3:** (1) `mask ^= 1 << node.val`. (2) `if not node.left and not node.right and mask & (mask - 1) == 0`. (3) `stack.append((c, mask))`.

**Bài 4:** (1) Mang thêm đường đi (tuple) hoặc dùng list + quay lui. (2) Tại lá, kiểm tra mask. (3) `sorted(...)` khi trả về.

**Bài 5:** (1) Đặt `pre(v)` = mask đường gốc → `v`; mask đường `u → v` = `pre(v) ^ pre(cha(u))`. (2) Khi DFS, giữ dict đếm các `pre` của **tổ tiên** (kể cả mask rỗng 0 cho "cha của gốc"). (3) Tại `v`, cộng số tổ tiên có `pre = pre(v)` hoặc `pre(v) ^ (1 << d)` với `d = 1..9`; thêm `pre(v)` vào dict trước khi xuống con, bỏ ra khi quay lên.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
b1 = [1, 9, 1, None, 1, None, 1, None, None, 7, None, None, 4]
ans1 = Solution().pseudoPalindromicPaths(build_tree(b1))
print("Bài 1:", ans1, "| các đường:", root_to_leaf_paths(build_tree(b1)))
assert ans1 == pseudo_perm_brute(build_tree(b1)) == 1

# Bài 2
def leaf_masks(root):
    out, stack = [], [(root, 0)]
    while stack:
        node, mask = stack.pop()
        mask ^= 1 << node.val
        if not node.left and not node.right:
            out.append(sorted(d for d in range(1, 10) if mask >> d & 1))
        for c in (node.right, node.left):
            if c:
                stack.append((c, mask))
    return out

print("Bài 2:", leaf_masks(build_tree([2, 3, 1, 3, 1, None, 1])))
assert leaf_masks(build_tree([2, 3, 1, 3, 1, None, 1])) == [[2], [1, 2, 3], [2]]

# Bài 3
def pseudo_todo_sol(root):
    ans, stack = 0, [(root, 0)]
    while stack:
        node, mask = stack.pop()
        mask ^= 1 << node.val
        if not node.left and not node.right and mask & (mask - 1) == 0:
            ans += 1
        for c in (node.left, node.right):
            if c:
                stack.append((c, mask))
    return ans

for arr, want in examples + edge:
    assert pseudo_todo_sol(build_tree(arr)) == want
print("Bài 3: OK")

Bài 1: 1 | các đường: [[1, 9, 1], [1, 1, 1, 7, 4]]
Bài 2: [[2], [1, 2, 3], [2]]
Bài 3: OK


In [28]:
# Bài 4
def list_pseudo_paths(root):
    out, stack = [], [(root, 0, (root.val,))]
    while stack:
        node, mask, path = stack.pop()
        mask ^= 1 << node.val
        if not node.left and not node.right:
            if mask & (mask - 1) == 0:
                out.append(list(path))
        for c in (node.left, node.right):
            if c:
                stack.append((c, mask, path + (c.val,)))
    return sorted(out)

# Bài 5
def count_downward(root):
    seen = defaultdict(int)
    seen[0] = 1                          # pre của "cha của gốc"
    ans = 0
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 10**4))
    def dfs(node, pre):
        nonlocal ans
        pre ^= 1 << node.val
        ans += seen[pre] + sum(seen[pre ^ (1 << d)] for d in range(1, 10))
        seen[pre] += 1
        for c in (node.left, node.right):
            if c:
                dfs(c, pre)
        seen[pre] -= 1
    dfs(root, 0)
    return ans

def count_downward_brute(root):
    total = 0
    def starts(u):
        nonlocal total
        stack = [(u, 1 << u.val)]
        while stack:
            node, mask = stack.pop()
            total += mask & (mask - 1) == 0
            for c in (node.left, node.right):
                if c:
                    stack.append((c, mask ^ (1 << c.val)))
        for c in (u.left, u.right):
            if c:
                starts(c)
    starts(root)
    return total

assert list_pseudo_paths(build_tree([2, 3, 1, 3, 1, None, 1])) == [[2, 1, 1], [2, 3, 3]]
rng = random.Random(19)
for _ in range(400):
    tree = small_random_tree(rng.randint(1, 25), rng, rng.choice([[1, 2], [1, 2, 3], list(range(1, 10))]))
    paths = root_to_leaf_paths(tree)
    assert list_pseudo_paths(tree) == sorted(p for p in paths if has_pal_perm(p))
    assert count_downward(tree) == count_downward_brute(tree)
print("Bài 4:", list_pseudo_paths(build_tree([2, 3, 1, 3, 1, None, 1])))
print("Bài 5: count_downward(ví dụ 1) =", count_downward(build_tree([2, 3, 1, 3, 1, None, 1])), "| OK (400 cây)")

Bài 4: [[2, 1, 1], [2, 3, 3]]
Bài 5: count_downward(ví dụ 1) = 10 | OK (400 cây)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template bitmask chẵn lẻ trên đường gốc → lá:**

```python
ans, stack = 0, [(root, 0)]
while stack:
    node, mask = stack.pop()
    mask ^= 1 << node.val
    if not node.left and not node.right:
        ans += mask & (mask - 1) == 0      # <= 1 chữ số lẻ lần
    else:
        if node.left:  stack.append((node.left, mask))
        if node.right: stack.append((node.right, mask))
return ans
```

**Checklist:**
- [ ] Điều kiện là "≤ 1 chữ số lẻ lần", không phải "mọi chữ số chẵn".
- [ ] Chỉ đếm ở **lá thật** (không có cả hai con), không đếm ở `None`.
- [ ] Lật bit đúng: `mask ^= 1 << val`.
- [ ] Nếu dùng mảng đếm chung: có **quay lui**.
- [ ] Đã nghĩ tới cây lệch `10^5` tầng; đã thử cây 1 nút, nút một con.